1. Carga del dataset

In [1]:
import pandas as pd

In [2]:
df_original = pd.read_csv("../Data/Raw/data_act_01.csv", sep=';')

In [3]:
df_original.head()

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,Range,AddressType
0,160903280,Assault / Battery,2016-03-30T00:00:00,18:42,2016-03-30T18:42:00,REP,100 Block Of Chilton Av,San Francisco,CA,1,NaN,Premise Address
1,160912272,Homeless Complaint,2016-03-31T00:00:00,15:31,2016-03-31T15:31:00,GOA,2300 Block Of Market St,San Francisco,CA,1,NaN,Premise Address
2,160912590,Susp Info,2016-03-31T00:00:00,16:49,2016-03-31T16:49:00,GOA,2300 Block Of Market St,San Francisco,CA,1,NaN,Premise Address
3,160912801,Report,2016-03-31T00:00:00,17:38,2016-03-31T17:38:00,GOA,500 Block Of 7th St,San Francisco,CA,1,NaN,Premise Address
4,160912811,594,2016-03-31T00:00:00,17:42,2016-03-31T17:42:00,REP,Beale St/bryant St,San Francisco,CA,1,NaN,Intersection


2. Inspeccionando el dataset

In [4]:
df_original.shape


(10051, 12)

In [5]:
df_original.describe()

,CrimeId,Range
count,1.005100e+04,0.0
mean,1.609394e+08,NaN
std,1.327006e+04,NaN
min,1.609033e+08,NaN
25%,1.609303e+08,NaN
50%,1.609408e+08,NaN
75%,1.609513e+08,NaN
max,1.609642e+08,NaN


In [6]:
df_original.describe(include="all")

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,Range,AddressType
count,1.005100e+04,10051,10051,10051,10051,10051,10051,9730,10048,10051,0.0,10051
unique,NaN,575,9,1416,5116,19,5387,8,1,2,NaN,6
top,NaN,Traffic Stop,2016-04-02T00:00:00,17:39,2016-04-01T17:21:00,HAN,900 Block Of Market St,San Francisco,CA,1,NaN,Premise Address
freq,NaN,1215,2259,19,8,2820,58,9665,10048,10048,NaN,5059
mean,1.609394e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.327006e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.609033e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,1.609303e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.609408e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.609513e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CrimeId                10051 non-null  int64  
 1   OriginalCrimeTypeName  10051 non-null  str    
 2   OffenseDate            10051 non-null  str    
 3   CallTime               10051 non-null  str    
 4   CallDateTime           10051 non-null  str    
 5   Disposition            10051 non-null  str    
 6   Address                10051 non-null  str    
 7   City                   9730 non-null   str    
 8   State                  10048 non-null  str    
 9   AgencyId               10051 non-null  str    
 10  Range                  0 non-null      float64
 11  AddressType            10051 non-null  str    
dtypes: float64(1), int64(1), str(10)
memory usage: 942.4 KB


3. Filas movidas

Tal y como hemos visto con el método df.info(), se identificó una anomalía de tipo en la columna AgencyId. Mientras que un identificador de agencia debería ser de naturaleza numérica (int64), Pandas categorizó la columna como str (string). Si hubiesemos intentado transformar esa columna a numerica, no nos hubiera dejado hacerla, nos hubiera dado errror. Esto indicó la presencia de valores alfanuméricos inesperados que contaminaban la dimensión.

In [8]:
# Intentamos transformar la columna AgencyId a numérica

try:
    df_original['AgencyId'] = pd.to_numeric(df_original['AgencyId'])
except:
    print("Dicha columna no pudo ser transformada a numérica")

Dicha columna no pudo ser transformada a numérica


In [9]:
print(df_original['AgencyId'].unique())

<StringArray>
['1', 'CA']
Length: 2, dtype: str


Viendo los valores únicos de dicha columna, observamos que un valor si es númerico 1 pero el otro es CA, lo que nos hace sospechar que ese valor sea de la anterior columna (State) que por algún motivo, se haya desplazado. Veamos las filas donde dicha columna es CA para ver que esta ocurriendo.

In [10]:
df_original[df_original['AgencyId'] == 'CA']

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,Range,AddressType
5771,160942112,Auto Boost / Strip,2016-04-03T00:00:00,14:30,2016-04-03T14:30:00,REP,Martin Luther King Dr/bowling Green Dr,NaN,NaN,CA,NaN,1
8021,160952280,Auto Boost / Strip,2016-04-04T00:00:00,14:46,2016-04-04T14:46:00,REP,Martin Luther King Dr/nancy Pelosi Dr,S,NaN,CA,NaN,1
8473,160953118,Auto Boost / Strip,2016-04-04T00:00:00,18:11,2016-04-04T18:11:00,REP,Conservatory Drive E/john F Kennedy Dr,NaN,NaN,CA,NaN,1


Con esto, se confirma una correlación directa entre el valor anómalo en AgencyId y la ausencia de datos en la columna State. Esta evidencia valida la hipótesis de un desplazamiento de campos.

Vamos a reubicar el valor geográfico 'CA' en su dimensión correspondiente (State) y se restaura el identificador de agencia al valor canónico '1'. Esta acción garantiza la recuperación de la información sin comprometer la consistencia de tipos necesaria para las fases posteriores de análisis numérico.

In [11]:
# 1. Identificamos las filas donde AgencyId es 'CA'
mask = df_original['AgencyId'] == 'CA'

# 2. Corregimos: En esas filas, ponemos el State como 'CA' y el AgencyId como '1'
df_original.loc[mask, 'State'] = 'CA'
df_original.loc[mask, 'AgencyId'] = '1'

In [12]:
# Transformamos el tipo de AgencyId a numérico (int)
df_original['AgencyId'] = pd.to_numeric(df_original['AgencyId'])

In [13]:
# Comprobemos ahora que los tipos de cada columna tengan coherencia con su contenido
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CrimeId                10051 non-null  int64  
 1   OriginalCrimeTypeName  10051 non-null  str    
 2   OffenseDate            10051 non-null  str    
 3   CallTime               10051 non-null  str    
 4   CallDateTime           10051 non-null  str    
 5   Disposition            10051 non-null  str    
 6   Address                10051 non-null  str    
 7   City                   9730 non-null   str    
 8   State                  10051 non-null  str    
 9   AgencyId               10051 non-null  int64  
 10  Range                  0 non-null      float64
 11  AddressType            10051 non-null  str    
dtypes: float64(1), int64(2), str(9)
memory usage: 942.4 KB


4. Valores nulos

In [14]:
df_original.isnull().sum()

CrimeId                      0
OriginalCrimeTypeName        0
OffenseDate                  0
CallTime                     0
CallDateTime                 0
Disposition                  0
Address                      0
City                       321
State                        0
AgencyId                     0
Range                    10051
AddressType                  0
dtype: int64


Como se puede observar, la columna rango tiene todo sus valores nulos.... Como esto no nos da información relevante, podemos eliminar dicha columna. Además hay 321 ciudades que tiene valores faltantes. Podemos llenar estos valores con el string "Unknown" para no descartar tantos registros.

In [15]:
# Eliminamos columna Range por alto porcentaje de nulo

df_no_range_cityUnknown = df_original.drop(columns=["Range"])


In [16]:
# Sustituimos los nulos de la columna 'City' por 'Unknown'
df_no_range_cityUnknown['City'] = df_no_range_cityUnknown['City'].fillna('Unknown')

In [17]:
df_no_range_cityUnknown.isnull().sum()

CrimeId                  0
OriginalCrimeTypeName    0
OffenseDate              0
CallTime                 0
CallDateTime             0
Disposition              0
Address                  0
City                     0
State                    0
AgencyId                 0
AddressType              0
dtype: int64

5. Filas duplicadas

In [18]:
print(f"Cantidad de filas duplicadas: {df_no_range_cityUnknown.duplicated().sum()}")

Cantidad de filas duplicadas: 0


6. Duplicados por columnas incongruentes

En nuestro dataset, tenemos que cada crimen deberia tener un ID único. Por eso mismo la columna CrimeId no deberia tener duplicados. Comprobemoslo:

In [19]:
df_no_range_cityUnknown['CrimeId'].value_counts()

CrimeId
160913455    3
160950496    3
160903280    1
160912272    1
160912590    1
            ..
160964210    1
160964216    1
160964227    1
160964229    1
160964249    1
Name: count, Length: 10047, dtype: int64

Observamos que hay 3 filas con ID 160913455 y otras 3 filas con ID  160950496. Anteriormente hemos visto que no habian filas duplicadas. Asi que hay 3 crimenes diferentes con el mismo ID 160913455 y otros 3 crimenes diferentes con el mismo ID 160950496. 

In [20]:
# 1. Calculamos las frecuencias de cada ID
counts = df_no_range_cityUnknown['CrimeId'].value_counts()

# 2. Filtramos solo los IDs que aparecen más de una vez
ids_duplicados = counts[counts > 1].index

# 3. Mostramos todas las filas cuyo CrimeId esté en esa lista de repetidos
display(df_no_range_cityUnknown[df_no_range_cityUnknown['CrimeId'].isin(ids_duplicados)].sort_values(by='CrimeId'))

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,AddressType
26,160913455,Vandalism,2016-03-31T00:00:00,20:53,2016-03-31T20:53:00,ND,1600 Block Of Sunnydale Av,San Francisco,CA,1,Premise Address
1707,160913455,Susp,2016-04-01T00:00:00,18:29,2016-04-01T18:29:00,GOA,Geary St/larkin St,San Francisco,CA,1,Intersection
3792,160913455,Passing Call,2016-04-02T00:00:00,17:11,2016-04-02T17:11:00,Not recorded,900 Block Of Market St,San Francisco,CA,1,Premise Address
7045,160950496,Passing Call,2016-04-04T00:00:00,6:51,2016-04-04T06:51:00,HAN,University St/felton St,San Francisco,CA,1,Intersection
7046,160950496,Suspicious Vehicle,2016-04-04T00:00:00,6:51,2016-04-04T06:51:00,ND,1400 Block Of Cabrillo St,San Francisco,CA,1,Premise Address
7047,160950496,Trespasser,2016-04-04T00:00:00,6:51,2016-04-04T06:51:00,CAN,Block Of Hampshire St,San Francisco,CA,1,Premise Address


Tras inspeccionar los registros con el CrimeId duplicado, hemos comprobado que estos casos no correspondían a registros duplicados reales, sino a incidentes distintos que, por inconsistencias en la fuente original, fueron etiquetados con el mismo identificador. Durante el análisis se observaron dos patrones principales:
- Agrupación de incidentes bajo un mismo ID, que diferían en tipo de delito, fecha u otros atributos relevantes.
- Agrupación de múltiples incidentes simultáneos ocurridos en ubicaciones distintas, lo que confirma que no representan el mismo evento.

Tras evaluar distintas alternativas y con el objetivo de garantizar la integridad del modelo de datos, he optado por mantener el CrimeId original tal cual, incluso cuando presenta duplicados. La razón es que generar un nuevo identificador artificial (por ejemplo, asignando valores consecutivos libres como 160913456, 160913457, etc.) podría provocar colisiones futuras si la fuente de datos incorporase nuevos registros con esos mismos valores.
De este modo, si en algún punto del análisis se requiere un identificador único y estable, se podría usar el índice propio del dataframe, que garantiza unicidad sin alterar los datos originales ni introducir posibles conflictos futuros.



7. Formateo y extensión de columnas

Tal y como vimos con la función info(), las columnas OffenseDate y CallDateTime son de tipo string en vez de tipo datetime que seria lo lógico para fechas. Asi pues el primer paso es transformar dichas columnas al tipo correcto.

In [21]:
df_no_range_cityUnknown["OffenseDate"] = pd.to_datetime(df_no_range_cityUnknown["OffenseDate"])
df_no_range_cityUnknown["CallDateTime"] = pd.to_datetime(df_no_range_cityUnknown["CallDateTime"])

In [22]:
df_no_range_cityUnknown.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   CrimeId                10051 non-null  int64         
 1   OriginalCrimeTypeName  10051 non-null  str           
 2   OffenseDate            10051 non-null  datetime64[us]
 3   CallTime               10051 non-null  str           
 4   CallDateTime           10051 non-null  datetime64[us]
 5   Disposition            10051 non-null  str           
 6   Address                10051 non-null  str           
 7   City                   10051 non-null  str           
 8   State                  10051 non-null  str           
 9   AgencyId               10051 non-null  int64         
 10  AddressType            10051 non-null  str           
dtypes: datetime64[us](2), int64(2), str(7)
memory usage: 863.9 KB


Para facilitar el análisis estadístico y la visualización por periodos, extraemos el año, el mes y el día de la semana a nuevas columnas independientes. Esto nos permite segmentar los datos de forma eficiente para responder preguntas clave, como identificar los meses con mayor índice de criminalidad o determinar qué días de la semana presentan una mayor actividad de incidentes o qué año hubieron más crimenes o qué dias de la semana o que meses se registran mayor número de incidencias. Tambien se extrae los segundos que transcurren desde el incidente hasta que se notifica para poder preguntarnos si las incidencias tardan mucho en notificarse o no. En nuestro caso, este dato no es del todo real, ya que la hora de la llamada si se registra correctamente en el sistema pero la hora del suceso no. Para ello se pone un 0:00 del dia. Asi pues si un delito se cometio a las 18:29 y se notifico a las 19:05 estariamos registrando un retraso de 19 horas y 5 minutos en lugar de 36 minutos que es el retraso real. Aún así, considero que debe añadirse porque permite observar patrones de registros administrativos que revelan cómo se gestiona la información en el sistema, además de facilitar la detección de casos extremos donde el reporte se produzca con más de 24 o 48 horas de diferencia, lo cual suele ser un indicador clave de la naturaleza del incidente. Por último, porque eliminar variables debido a imprecisiones sistemáticas conocidas restaría profundidad y honestidad al estudio, mientras que conservarlas permite realizar una interpretación crítica de los resultados durante el análisis exploratorio, demostrando un control exhaustivo sobre las limitaciones de la fuente de datos. Finalmnete, tambien se crea una columna adicional sobre la hora (sin los minutos) se registra cada incidencia. De esta forma, podremos observar que periodo de tiempo se registran mayor incidencias, lo cual es útil para saber en que horas se precisa de mayor número de personal para poder ser atendidas con rapidez.

In [23]:
df_no_range_cityUnknown['Call_Year'] = df_no_range_cityUnknown['CallDateTime'].dt.year
df_no_range_cityUnknown['Call_Month'] = df_no_range_cityUnknown['CallDateTime'].dt.month
df_no_range_cityUnknown['Call_DayName'] = df_no_range_cityUnknown['CallDateTime'].dt.day_name()
df_no_range_cityUnknown['Offense_Year'] = df_no_range_cityUnknown['OffenseDate'].dt.year
df_no_range_cityUnknown['Offense_Month'] = df_no_range_cityUnknown['OffenseDate'].dt.month
df_no_range_cityUnknown['Offense_DayName'] = df_no_range_cityUnknown['OffenseDate'].dt.day_name()
df_no_range_cityUnknown['ReportingDelay'] = (df_no_range_cityUnknown['CallDateTime'] - df_no_range_cityUnknown['OffenseDate']).dt.total_seconds()
df_no_range_cityUnknown['Call_Hour'] = df_no_range_cityUnknown['CallTime'].str.split(':').str[0].astype(int)



8. Fechas inconsistentes

Este dataset contiene registros de incidentes reportados hasta el año 2016. Por lo tanto, cualquier registro con una fecha posterior a dicho año se consideraría una anomalía o un error de entrada de datos. Procederemos a identificar y tratar estas inconsistencias.

In [24]:
df_no_range_cityUnknown['Offense_Year'].value_counts()

Offense_Year
2016    10049
2013        1
2025        1
Name: count, dtype: int64

Observamos como la columna Call_Year tiene valores congruentes (todos del 2016), mientras que en la columna Offense_Year parece tener dos valores incongruentes. Por un lado tenemos un año aislado (2013) que podría ser correcto (por ejemplo, el suceso se produjese en aquel año pero hasta 2016 la policia no tuviese constancia de ello, hay delitos que no se informan al momento) y por otro un año futurista (2025) para dicho dataset. Veamos dichos registros.

In [25]:
df_no_range_cityUnknown[df_no_range_cityUnknown['Offense_Year'].isin([2013, 2025])]


,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,AddressType,Call_Year,Call_Month,Call_DayName,Offense_Year,Offense_Month,Offense_DayName,ReportingDelay,Call_Hour
7256,160950844,Traffic Stop,2013-04-04,9:00,2016-04-04 09:00:00,CIT,19th Av/lincoln Wy,San Francisco,CA,1,Intersection,2016,4,Monday,2013,4,Thursday,94726800.0,9
7925,160952096,Passing Call,2025-04-04,14:04,2016-04-04 14:04:00,HAN,Ida B Well High School,Unknown,CA,1,Geo-Override,2016,4,Monday,2025,4,Friday,-283946160.0,14


Analizandolo cautelosamente podemos observar que ambos registros inconsistentes comparten el mismo día y mes (04/04) tanto del OffenseDate como el de CallDateTime, variando solo el año. Dado que el registro de 2025 es un error evidente y el de 2013 es un valor extremo (outlier) que no representa el año 2016 del dataset, he decidido prescindir de ambos para no sesgar las visualizaciones temporales. 

In [26]:
# 1. Identificamos los índices de los registros molestos
indices_a_eliminar = df_no_range_cityUnknown[df_no_range_cityUnknown['Offense_Year'].isin([2013, 2025])].index

# 2. Los borramos del DataFrame
df_no_range_cityUnknown.drop(indices_a_eliminar, inplace=True)

# Reajustamos el índice para que no falten números
df_no_range_cityUnknown.reset_index(drop=True, inplace=True)

In [27]:
df_no_range_cityUnknown['Offense_Year'].value_counts()

Offense_Year
2016    10049
Name: count, dtype: int64

In [28]:
df_no_range_cityUnknown['Call_Year'].value_counts()

Call_Year
2016    10049
Name: count, dtype: int64

Con esto, vemos que ahora "Offense_Year" y "Call_Year" solo tienen valores 2016. Esto nos genera un poco de ruido, es tener datos innecesarios en nuestro dataset en este caso. Si tuvieramos diferentes valores, como 2015, 2016 y 2017 tendria sentido para hacernos preguntas interesantes como ¿Qué año ha tenido un mayor indice de criminalidad? pero en nuestro caso como solo tiene un único valor, vamos a eliminar dicha columna.

In [29]:
# Eliminamos las columnas constantes
df_no_range_cityUnknown.drop(['Offense_Year', 'Call_Year'], axis=1, inplace=True)

Finalmente, verifiquemos que la hora que aparece en CallDateTime coincida exactamente con la hora registrada en la columna original CallTime. Así nos aseguramos de que no ha habido errores durante la conversión de los datos antes de borrar la columna que ya no necesitaremos.

In [30]:
# 1. Convertimos la hora de CallDateTime a texto en formato HH:MM (sin segundos), 
# y eliminamos ceros iniciales ya que en la columna CallTime no los hay, aparece como 9:00 y no como 09:00
hora_formateada = df_no_range_cityUnknown['CallDateTime'].dt.strftime('%H:%M').str.lstrip('0')

# 2. Hacemos lo mismo con CallTime por si acaso tiene espacios o ceros iniciales
hora_original = df_no_range_cityUnknown['CallTime'].str.strip().str.lstrip('0')

# 3. Buscamos las filas donde NO coinciden
discrepancias = df_no_range_cityUnknown[hora_formateada != hora_original]

# Mostramos cuántas hay
print(f"Número de discrepancias encontradas: {len(discrepancias)}")


Número de discrepancias encontradas: 1


Tras realizar la comprobación, se ha detectado una única discrepancia que no coincide. Veamosla:

In [31]:
discrepancias

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,AddressType,Call_Month,Call_DayName,Offense_Month,Offense_DayName,ReportingDelay,Call_Hour
2887,160931014,Dw,2016-04-02,22:31,2016-04-02 09:31:00,CIT,800 Block Of 47th Av,San Francisco,CA,1,Premise Address,4,Saturday,4,Saturday,34260.0,22


Asi pues podemos observar que la hora de CallTime (22:31) no coincide con la de CallDateTime (09:31). Ante la imposibilidad de determinar cuál es el dato correcto, se opta por eliminar este registro para garantizar la máxima precisión en el análisis temporal posterior.

In [32]:
# Eliminamos la fila de la discrepancia usando su índice
df_no_range_cityUnknown.drop(discrepancias.index, inplace=True)

# Reseteamos los índices
df_no_range_cityUnknown.reset_index(drop=True, inplace=True)

In [33]:
# 1. Convertimos la hora de CallDateTime a texto en formato HH:MM (sin segundos), 
# y eliminamos ceros iniciales ya que en la columna CallTime no los hay, aparece como 9:00 y no como 09:00
hora_formateada = df_no_range_cityUnknown['CallDateTime'].dt.strftime('%H:%M').str.lstrip('0')

# 2. Hacemos lo mismo con CallTime por si acaso tiene espacios o ceros iniciales
hora_original = df_no_range_cityUnknown['CallTime'].str.strip().str.lstrip('0')

# 3. Buscamos las filas donde NO coinciden
discrepancias = df_no_range_cityUnknown[hora_formateada != hora_original]

# Mostramos cuántas hay
print(f"Número de discrepancias encontradas: {len(discrepancias)}")


Número de discrepancias encontradas: 0


Finalmente, verifiquemos la consistencia cronológica de los eventos. No es lógicamente posible que una llamada de reporte (CallDateTime) ocurra antes de la comisión del delito (OffenseDate). Para ello, comprobemos que no existen valores negativos en la diferencia (dado en la columna ReportingDelay).

In [34]:
df_no_range_cityUnknown[df_no_range_cityUnknown['ReportingDelay'] < 0]

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,AddressType,Call_Month,Call_DayName,Offense_Month,Offense_DayName,ReportingDelay,Call_Hour


Efectivamente, observamos que no hay registros. Asi pues, significa que todos los registros siguen una línea temporal coherente.

9. Estandarización

A continuación procederemos a estandarizar las columnas OriginalCrimeTypeName y City para asegurar la consistencia de los datos. Actualmente, el sistema podría interpretar 'San Francisco' y 'SAN FRANCISCO' como ciudades distintas debido a la diferencia de mayúsculas. Para evitar esta duplicidad y asegurar que los conteos sean exactos, transformaremos todos los valores a un formato único y eliminaremos espacios en blanco innecesarios. Empecemos por el de City.

In [35]:
df_no_range_cityUnknown['City'].value_counts()

City
San Francisco    9663
Unknown           320
Treasure Isla      51
Daly City           5
Yerba Buena         3
Presidio            3
SAN FRANCISCO       1
 S                  1
Brisbane            1
Name: count, dtype: int64

Para empezar, vemos que tenemos un "San Francisco" y un "SAN FRANCISCO". Pongamoslo como "San Francisco".

In [36]:
# Pasamos todo a formato título (San Francisco) y quitamos espacios
df_no_range_cityUnknown['City'] = df_no_range_cityUnknown['City'].str.title().str.strip()

Luego, podemos observar que tenemos como City "Presidio", "Yerba Buena" y "Treasure Isla". Esto no son ciudades como tal. Presidio es un parque de San Francisco, Yerba Buena es una zona militar de San Francisco y Treasure Island es una isla que pertenece a San Francisco. Es por ello que he decidido sustituir el valor de estas ciudades a "San Francisco".

In [37]:
# Creamos un diccionario con los cambios
mapeo_ciudades = {
    'Presidio': 'San Francisco',
    'Yerba Buena': 'San Francisco',
    'Treasure Isla': 'San Francisco'
}

# Aplicamos los cambios
df_no_range_cityUnknown['City'] = df_no_range_cityUnknown['City'].replace(mapeo_ciudades)

Finalmente, tenemos un registro en la ciudad con valor 'S'. Podemos intuir que se refiere a "San Francisco" pero aseguremonos viendo el Address de dicho registro.

In [38]:
city_S = df_no_range_cityUnknown[df_no_range_cityUnknown['City'] == 'S']

print(city_S['Address'])

8018    Martin Luther King Dr/nancy Pelosi Dr
Name: Address, dtype: str


Tras analizar su dirección (Address), se confirma que se ubica en la intersección de Martin Luther King Dr y Nancy Pelosi Dr, dentro del Golden Gate Park de San Francisco. Para asegurar la integridad de los datos, he decidido corregir este valor, reemplazando "S" a "San Francisco".

In [39]:
df_no_range_cityUnknown['City'] = df_no_range_cityUnknown['City'].replace({'S': 'San Francisco'})

In [40]:
df_no_range_cityUnknown['City'].value_counts()

City
San Francisco    9722
Unknown           320
Daly City           5
Brisbane            1
Name: count, dtype: int64

Veamos ahora el de OriginalCrimeTypeName.

In [41]:
df_no_range_cityUnknown['OriginalCrimeTypeName'].value_counts()

OriginalCrimeTypeName
Traffic Stop          1214
Passing Call          1110
Suspicious Person      685
Homeless Complaint     585
22500e                 359
                      ... 
Phys                     1
222 Att                  1
Wireless-Open            1
Adv To Co A              1
Wireless-H/u             1
Name: count, Length: 575, dtype: int64

Debido a la enorme diversidad de valores únicos, resulta imposible revisar manualmente cada nombre para identificar crímenes idénticos. Por ello, he decidido a aplicar una normalización de formato, convirtiendo los textos a tipo 'Título' (primera letra en mayúscula) y eliminando espacios en blanco sobrantes para mejorar la consistencia visual del dataset.

In [42]:
# Pasamos todo a formato título y quitamos espacios
df_no_range_cityUnknown['OriginalCrimeTypeName'] = df_no_range_cityUnknown['OriginalCrimeTypeName'].str.title().str.strip()

In [43]:
df_no_range_cityUnknown['OriginalCrimeTypeName'].value_counts()

OriginalCrimeTypeName
Traffic Stop          1214
Passing Call          1110
Suspicious Person      685
Homeless Complaint     585
22500E                 359
                      ... 
Phys                     1
222 Att                  1
Wireless-Open            1
Adv To Co A              1
Wireless-H/U             1
Name: count, Length: 575, dtype: int64

10. Otras comprobaciones

Hay tres columnas, que no hemos revisado que esten correctamente. Son el caso de Disposition, AddressType y State. Empecemo con el State.

In [44]:
df_no_range_cityUnknown['State'].value_counts()

State
CA    10048
Name: count, dtype: int64

En este caso, vemos que no hay ninguna incongruencia. Pasemos ahora con Disposition.

In [45]:
df_no_range_cityUnknown['Disposition'].value_counts()

Disposition
HAN             2819
CIT             1423
GOA             1273
ADV             1142
REP              800
Not recorded     543
ND               427
UTL              385
CAN              353
NOM              324
PAS              170
ABA               97
NCR               82
22                77
ARR               65
ADM               48
INC               17
CRT                2
SFD                1
Name: count, dtype: int64

Al revisar dicha columna, podemos observar que la mayoría de los registros utilizan códigos de tres letras (como HAN o CIT). Sin embargo, se han detectado 77 registros con el valor numérico '22'. Este código corresponde a una terminación de cancelación en el sistema de radio de la policía. Se ha decidido mantenerlo como una categoría independiente para no perder la distinción original de estos incidentes. Finalmente, veamos que pasa con el AddressType.

In [46]:
df_no_range_cityUnknown["AddressType"].value_counts()

AddressType
Premise Address    5058
Intersection       3700
Common Location     818
Geo-Override        468
1                     3
Intersectioon         1
Name: count, dtype: int64

Con esto, hemos podido detectar una inconsistencia en el valor 'Intersectioon', el cual es una errata de 'Intersection'. Por otro lado, se identificó un valor numérico '1'. Si bien este registro es atípico para un campo de tipo de dirección, ante la falta de documentación específica que aclare su significado o confirme que se trata de un error, se ha decidido mantenerlo en su estado original para no sesgar el análisis. Esta decisión responde a un criterio de integridad de datos: no se debe transformar una variable sin la certeza de su origen.

In [47]:
# Veamos de todas formas los registros con AddressType '1' para ver si podemos sacar algo en claro
df_no_range_cityUnknown[df_no_range_cityUnknown["AddressType"] == '1']

,CrimeId,OriginalCrimeTypeName,OffenseDate,CallTime,CallDateTime,Disposition,Address,City,State,AgencyId,AddressType,Call_Month,Call_DayName,Offense_Month,Offense_DayName,ReportingDelay,Call_Hour
5770,160942112,Auto Boost / Strip,2016-04-03,14:30,2016-04-03 14:30:00,REP,Martin Luther King Dr/bowling Green Dr,Unknown,CA,1,1,4,Sunday,4,Sunday,52200.0,14
8018,160952280,Auto Boost / Strip,2016-04-04,14:46,2016-04-04 14:46:00,REP,Martin Luther King Dr/nancy Pelosi Dr,San Francisco,CA,1,1,4,Monday,4,Monday,53160.0,14
8470,160953118,Auto Boost / Strip,2016-04-04,18:11,2016-04-04 18:11:00,REP,Conservatory Drive E/john F Kennedy Dr,Unknown,CA,1,1,4,Monday,4,Monday,65460.0,18


Como se ve nada raro, simplemente camnbiaremos "Intersectioon" por "Intersection" y dejaremos este "1" tal cual esta. Si fuese un proyecto real, habria que investigar que significa ese "1", si tiene algun sentido como el "22" de Disposition.

In [48]:
# Corregir el error de dedo
df_no_range_cityUnknown['AddressType'] = df_no_range_cityUnknown['AddressType'].replace('Intersectioon', 'Intersection')

11. Exportación

Finalmente exportamos el dataset limpio a un CSV

In [49]:
df_no_range_cityUnknown.to_csv("../Data/Cleaned/data_cleaned.csv", index=False)